# Certify fault tolerance

This notebook shows how to compute exact certificates of fault tolerance for noisy Stim circuits.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim
import sympy

from qldpc import circuits, codes

## Certify the fault distance of a memory experiment

The fault distance of a noisy circuit is the smallest number of faults that flip a logical observable without firing any detector.
Enumerating every combination of faults is impractical beyond small circuits: the bivariate bicycle example below has 432 faults after merging equivalent error mechanisms, so checking every combination of up to 6 of them would take $\sum_{k=1}^{6} \binom{432}{k} \approx 9 \times 10^{12}$ steps.
`circuits.get_fault_distance` instead converts the circuit into an auxiliary CSS code and uses the Brouwer-Zimmermann code-distance algorithm, which stops as soon as it can prove that no lighter undetected fault exists.
`circuits.get_fault_distance` thereby computes the fault distance exactly in a few seconds here, and it scales to much larger circuits, though its cost is still exponential in the fault distance.
See `help(circuits.get_fault_distance)` for additional information.

Here we compute the fault distance of X- and Z-basis memory experiments for the [[72,12,6]] bivariate bicycle code from [arXiv:2308.07915](https://arxiv.org/abs/2308.07915), with one round of syndrome extraction under circuit-level depolarizing noise.

In [3]:
x, y = sympy.symbols("x y")
code = codes.BBCode({x: 6, y: 6}, x**3 + y + y**2, y**3 + x + x**2)
noise_model = circuits.DepolarizingNoiseModel(1e-3)


def get_noisy_memory_circuit(code, basis, num_rounds=1):
    # one logical memory cycle = num_rounds rounds of syndrome extraction
    # you can replace logical_memory_cycle with another memory circuit
    initialization, logical_memory_cycle, readout, *_ = circuits.get_memory_experiment_parts(
        code, basis=basis, num_rounds=num_rounds
    )

    # keep initialization and readout noiseless, to certify only the inner QEC cycle
    return initialization + noise_model.noisy_circuit(logical_memory_cycle) + readout


print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)

    # the number of detectors flipped by each fault of the detector error model
    detectors_per_fault = [
        sum(target.is_relative_detector_id() for target in instruction.targets_copy())
        for instruction in noisy_circuit.detector_error_model().flattened()
        if instruction.type == "error"
    ]
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    print(
        f"{basis}-basis memory: {len(detectors_per_fault)} faults, "
        f"each flipping up to {max(detectors_per_fault)} detectors, "
        f"fault distance {fault_distance}"
    )

code parameters: (72, 12, 6)
X-basis memory: 432 faults, each flipping up to 6 detectors, fault distance 6
Z-basis memory: 432 faults, each flipping up to 6 detectors, fault distance 6


### The Steane code

The above example uses bare-ancilla syndrome extraction.
However, this syndrome measurement strategy does not always preserve the distance of a code.
In the case of the Steane code, the fault distance of the bare-ancilla syndrome measurement strategy is less than the distance of the Steane code, indicating that other (e.g., flagged, Shor, or Steane-type QEC) strategies are necessary for fault-tolerant QEC.

In [4]:
code = codes.SteaneCode()

print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)
    print(f"{basis}-basis memory: fault distance {circuits.get_fault_distance(noisy_circuit)}")

code parameters: (7, 1, 3)
X-basis memory: fault distance 2
Z-basis memory: fault distance 2


## Certify bounded fault propagation of a state-preparation circuit

`circuits.certify_state_prep_fault_propagation` checks that every set of faults leaves an output error that is equivalent, up to a stabilizer of the prepared state, to an error with weight at most the number of faults.
It suffices to check each fault on its own, because weight is subadditive.
See `help(circuits.certify_state_prep_fault_propagation)` for additional information.

Here we prepare GHZ states by fanning out CX gates, and add circuit-level depolarizing noise with a noise model.

In [5]:
noise_model = circuits.DepolarizingNoiseModel(1e-3)

for num_qubits in (3, 4):
    ghz_circuit = stim.Circuit("H 0")
    for target in range(1, num_qubits):
        ghz_circuit.append("CX", [0, target])
    noisy_circuit = noise_model.noisy_circuit(ghz_circuit)

    certificate = circuits.certify_state_prep_fault_propagation(noisy_circuit)
    print(f"{num_qubits}-qubit GHZ state: certified = {certificate.certified}")
    if witness := certificate.counterexample:
        (fault,) = witness.fault_indices
        print(f"  fault {fault} leaves an error equivalent to {witness.output_pauli}")
        print(f"  equivalent errors have weight at least {witness.effective_weight_lower_bound}")

3-qubit GHZ state: certified = True
4-qubit GHZ state: certified = False
  fault 4 leaves an error equivalent to +ZXX_
  equivalent errors have weight at least 2


With four qubits, one fault already leaves an error that no stabilizer reduces to weight one, so the certificate fails and returns that fault as a counterexample.

### CSS weight versus physical weight

By default, a CSS state is judged by its CSS weight (the larger of the X weight and the Z weight) and any other state by its physical weight; `css_weight=` overrides this choice.

In [6]:
product_circuit = stim.Circuit("H 1\nDEPOLARIZE2(0.01) 0 1")  # |0>|+> with two-qubit noise
for css_weight in (None, False):
    certificate = circuits.certify_state_prep_fault_propagation(
        product_circuit, css_weight=css_weight
    )
    print(
        f"css_weight={css_weight}: CSS weight used = {certificate.css_weight}, "
        f"certified = {certificate.certified}"
    )

css_weight=None: CSS weight used = True, certified = True
css_weight=False: CSS weight used = False, certified = False


Two-qubit depolarizing noise can apply $X$ to the first qubit and $Z$ to the second, an error with CSS weight one but physical weight two.